# Using the UNIONS/CFIS ShapePipe weak-lensing catalogues

Tutorial for v1.0 (April 2022)

See also `README_SP_v1.0.txt`.

Author: Martin Kilbinger <martin.kilbinger@cea.fr>

ShapePipe core team:
- Axel Guinot
- Sam Farrens
- Tobias Liaudat

In [1]:
# Required python libraries
import numpy as np
from astropy.io import fits
import matplotlib as mpl
import matplotlib.pylab as plt

plt.rcParams['font.size'] = 20
%matplotlib inline

In [2]:
# Optional: To compute the example shear-shear correlation functions
import treecorr

In [3]:
# Catalogue name base
cat_base = 'unions_shapepipe_2022'

# Version
version = 'v1.0'

## 1. Base calibrated weak-lensing catalogue
This catalogue can be used *as is*.
The shear estimates are calibrated for the entire catalogue. If you intend to apply cuts, in particular to magnitude, the calibration should be redone. In that case we recommend to use the extended catalogue, see Section 2 below.

### Open FITS file

In [ ]:
cat_name = f'{cat_base}_{version}.fits'
print(f'Opening base catalogue {cat_name}...')
hdu_list = fits.open(cat_name)
data = hdu_list[1].data

### Print file content information

In [ ]:
# Column names
data.dtype.names

In [ ]:
# Number of objects
len(data)

In [ ]:
# Print headers
hdu_list[0].header

In [ ]:
hdu_list[1].header

### Example: Compute shear-shear correlation function

In [ ]:
# Create TreeCorr configuration

# Set unit of angular separation
sep_unit = 'arcmin'

TreeCorrConfig = {
        'ra_units': 'degrees',
        'dec_units': 'degrees',
        'max_sep': '200',
        'min_sep': '1',
        'sep_units': sep_unit,
        'nbins': 20
    }

In [ ]:
# Create TreeCorr catalogue.

# Get coordinate units from FITS file
coord_unit = hdu_list[1].header['TUNIT1']
if coord_unit != hdu_list[1].header['TUNIT2']:
    raise ValueError('Units for RA and Dec should be equal')

cat_gal = treecorr.Catalog(
    ra=data['RA'],
    dec=data['Dec'],
    g1=data['e1'],
    g2=data['e2'],
    w=data['w'],
    ra_units=coord_unit,
    dec_units=coord_unit
)

In [ ]:
# Create correlation object
gg = treecorr.GGCorrelation(TreeCorrConfig)

In [ ]:
# Process correlation. This can take a few minutes.
gg.process(cat_gal, cat_gal)

In [ ]:
# Plot components of the correlation functions
plt.figure(figsize=(14,8))

plt.errorbar(gg.meanr, gg.xip, yerr=np.sqrt(gg.varxip), label=r'$\xi_+$')
plt.errorbar(gg.meanr, gg.xim, yerr=np.sqrt(gg.varxim), label=r'$\xi_-$')
plt.loglog()

plt.legend()
plt.xlabel(rf'$\theta$ [{sep_unit}]')
_ = plt.ylabel(r'$\xi_\pm(\theta)|$')

## 2. Extended weak-lensing catalogue

This catalogue needs to be *calibrated before use*.
It contains the relevant information to apply the calibration.
Cuts can be applied to the catalogue before calibration.

### Open FITS file

In [ ]:
cat_ext_name = f'{cat_base}_extended_{version}.fits'
print(f'Openging catalogue {cat_ext_name}...')
hdu_list_ext = fits.open(cat_ext_name)
data_ext = hdu_list_ext[1].data 

### Print file content information

In [ ]:
# Print column names
data_ext.dtype.names

In [ ]:
# Number of objects
len(data_ext)

### Shear calibration

The calibration for ShapePipe depends on the properties of the galaxy sample. It should be
carried out after selecting a (sub-)sample of galaxies.

In [ ]:
# First step: Apply cuts, select (sub-)sample of galaxies.
# The following dummy example selects the entire sample.
mask = [True] * len(data_ext)
data_ext_sub = data_ext[mask]

In [ ]:
# Second step: Carry out calibration, as follows.

#### Compute additive bias

In [ ]:
# For a survey as large as UNIONS/CFIS, the assumption that the mean shear over
# the observed area vanishes, is a very good approximation.
# Then, the additive bias is the weighted mean of the uncalibrated ellipticities.

# This value corresponding to the entire sample is also given in the
# catalogue FITS header.

c = np.empty(shape=(2))
for comp in (0, 1):
    c[comp] = np.average(
        data_ext_sub[f'e{comp+1}_uncal'],
        weights=data_ext_sub['w']
    )

In [ ]:
print('Additive bias')
for comp in (0, 1):
    print(f'c_{comp+1} = {c[comp]:.3g}')

#### Compute multiplicative bias

In the metacalibration the multiplicative bias is a 2x2 matrix $R$.
It is the sum of the shear response matrix $R_g$ and the selection response matrix $R_\textrm{s}$,
$$R = R_g + R_\textrm{s} .$$
These matrices corresponding to the entire sample are also given in the catalogue FITS header.

##### Shear response matrix $R_g$

In [ ]:
# The shear response matrix for each individual galaxy is part of the extended catalogue.
# The corresponding ensemble matrix for a (sub-)sample needs to be computed
# after selecting the (sub-)sample in the first step above.

R_g = np.empty(shape=(2, 2))

for idx in (0, 1):
    for jdx in (0, 1):
        R_g[idx, jdx] = np.mean(data_ext_sub[f'R_g{idx+1}{jdx+1}'])

In [ ]:
print('Shear response matrix R_g =')
print(np.matrix(R_g))

##### Selection matrix $R_\textrm{s}$

The selection matrix was pre-computed for the entire galaxy sample; it is not
possible do obtain $R_\textrm{s}$ for individual galaxies. Therefore, in this
version of the ShapePipe catalogue we use the global selection matrix also for
sub-samples of galaxies. This will be improved in future versions.

Since $|R_\textrm{s}| < |R_g|$, the resulting error is small.

In [ ]:
R_s = np.empty(shape=(2, 2))
for idx in (0, 1):
    for jdx in (0, 1):
        R_s[idx][jdx] = hdu_list_ext[0].header[f'R_S{idx+1}{jdx+1}']

In [ ]:
print('Selection response matrix R_s =')
print(np.matrix(R_s))

##### Total response matrix $R = R_g + R_\textrm{s}$

In [ ]:
R = R_g + R_s

print('Total response matrix R =')
print(np.matrix(R))

#### Apply calibration to raw shear values and obtain calibrated shear estimates

In [ ]:
e_uncal_minus_c = np.array([
    data_ext['e1_uncal'] - c[0],
    data_ext['e2_uncal'] - c[1]
])
Rm1 = np.linalg.inv(R)
e_cal = Rm1.dot(e_uncal_minus_c)

In [ ]:
# Cross-check (*only* valid if no cut has been applied to galaxysample):
# Compare first few calibrated shears from base catalogue to
# the ones from the extended catalogue calibrated "by hand"

print(e_cal[:,0:3])

print(data['e1'][0:3], data['e2'][0:3])